# GITT analysis
In this notebook we will use cellpy to extract the open circuit voltages (OCV) from a GITT measurement. The extracted OCVs will be plotted, and the results saved in .csv format."

In [1]:
import pathlib
from cellpy import prms

prms.Paths.examplesdir = pathlib.Path("./data/examples")

In [2]:
import pandas as pd
import matplotlib.pyplot as plt


import cellpy
from cellpy.utils import plotutils

Set filepath and load the datafile:

In [4]:
filedir = pathlib.Path("data") / "raw"  # foldername within the same directory
c = cellpy.get(filedir / "20210210_FC.h5")

Produce an overview plot to identify cycle numbers for the GITT experiment (for an interactive version of this plot, you have to have `plotly` installed):

In [ ]:
plotutils.cycle_info_plot(c, cycle=list(range(2, 7)))

### OCV extraction
From the overview plot above, we can identify the GITT cycles to be cycle number 4 and 5. In the following, we will focus on cycle 5 only.

For further analysis, we create the **step table**, called  ``steps``, a dataframe that contains a lot ofnformation on all the cycle steps for the cell.

In the following, we apply several filters to ``steps``, to eventually extract OCV voltages and corresponding capacities:

1. **``steps_cycle``**: Extract the rows specifically for the selected GITT cycle (here: cycle Nr 5).


NB: For simplicity, ``steps_cycle`` only contains rows relevant for further analysis, i.e. *"cycle", "step""charge_last", "discharge_last", "voltage_first" ,"voltage_last", "type"*."

In [6]:
GITT_cycle = 5
c.make_step_table(all_steps=True)
steps = c.data.steps
steps_cycle = steps.loc[
    (steps.cycle == GITT_cycle),
    [
        "cycle",
        "step",
        "charge_last",
        "discharge_last",
        "voltage_first",
        "voltage_last",
        "type",
    ],
]

Taking a closer look at the created ``steps_cycle`` dataframe:

- `steps_cycle.head(10)` to view the first 10 rows
- `steps_cycle.tail(10)` to view the last 10 rows

In [ ]:
steps_cycle.tail(10)

2. To extract the OCV voltages, we then filter the `steps_cycle` dataframe for 
    - the OCV relaxation steps on discharge, ``steps_ocv_dch``, of type *oxvrlx_up* (and *rest*), corresponding to ``step==3``, and
    - the OCV relaxation steps on charge ``steps_ocv_cha``, of type *oxvrlx_down* (and *rest*), corresponding to ``step==8``.
Thereby we obtain two new dataframes

In [8]:
steps_ocv_cha = steps_cycle.loc[steps_cycle.step == 3]
steps_ocv_dch = steps_cycle.loc[steps_cycle.step == 8]

In [ ]:
steps_ocv_cha.head(5)

The voltages at the end of these steps (`voltage_last`), contain the (pseudo-) OCV voltages:

In [10]:
V_cha = steps_ocv_cha.voltage_last.reset_index(drop=True)
V_dch = steps_ocv_dch.voltage_last.reset_index(drop=True)
cap_cha = (
    steps_ocv_cha.charge_last.reset_index(drop=True) * 1000
)  # *1000 to convert to mAh
cap_dch = (
    steps_ocv_dch.discharge_last.reset_index(drop=True) * 1000
)  # *1000 to convert to mAh

To plot our results, we additionally get the entire voltage vs capacity curves for the selected GITT cycle, employing the `.get_ccap` and `.get_dcap` methods. The cell mass is used to convert from gravimetric capacity (mAh/g) to capacity (mAh).


In [11]:
c.make_step_table(all_steps=False)
ccap = c.get_ccap(cycle=GITT_cycle)
dcap = c.get_dcap(cycle=GITT_cycle)
mass = c.get_mass()  # in mg

In [ ]:
fig, ax = plt.subplots()
ax.plot(
    ccap["charge_capacity"] * mass / 1000, ccap["voltage"], color="blue", label="charge"
)
ax.plot(cap_cha, V_cha, "bo", label="OCV charge")
ax.plot(
    dcap["discharge_capacity"] * mass / 1000,
    dcap["voltage"],
    color="green",
    label="discharge",
)
ax.plot(cap_dch, V_dch, "go", label="OCV discharge")
plt.xlabel("Capacity [mAh]", fontsize=12)
plt.ylabel("Voltage [V]", fontsize=12)
plt.title("GITT OCV curve", fontsize=12)
# plt.ylim(0, 0.91)
# plt.xlim(0, 4.70)
ax.legend(fontsize=12)
fig.set_figheight(5)
fig.set_figwidth(10)
plt.show()

### Saving the data
Concatenate the OCV voltages and capacities into a dataframe, and save as a .csv file.

In [13]:
OCV_cha = pd.concat([cap_cha, V_cha], axis=1, keys=["Charge_cap_mAh", "OCV_V"])
OCV_dch = pd.concat([cap_dch, V_dch], axis=1, keys=["Discharge_cap_mAh", "OCV_V"])

In [14]:
outdir = pathlib.Path("out")

OCV_cha.to_csv(outdir / f"GITT_OCV_cycle{str(GITT_cycle)}_cha.csv", index=False)


OCV_dch.to_csv(outdir / f"GITT_OCV_cycle{str(GITT_cycle)}_dch.csv", index=False)

## Assignments
### A5.1 Redo for cycle number 4


### A5.2 Compare the two OCV curves (cycle 4 and 5)

### A5.3 Compare with C/20

Compare the voltage hysteresis obtained using C/20 cycling versus GITT OCV-curves